# The Law of Large Numbers

One measured person gives a noisy estimate of humanity's average height; ten give a better one; a thousand better still. The pattern has a name and a precise statement: the **law of large numbers** — as the sample size grows, the sample average tends to the population mean, $\frac1n\sum_{i=1}^n X_i \to \mu$ as $n \to \infty$, provided the samples are **i.i.d.** This notebook illustrates it with the video's four-sided-die experiment, states the conditions, and probes what happens when they fail. It is based on the fifth video of Lesson 5 (Sampling and Point Estimation).

## 1. Introduction

Want the average height of all humans, cheaply? Measure **one** person — that's an estimate, just noisy. Average **two or three** — slightly better. Ten, a hundred, a thousand — the more people, the better the estimate. And this happens for other metrics too, not just the expected value.

That empirical feeling — *more data, closer to the truth* — has been running underneath the whole module: unbiased estimators, shrinking standard errors, "bigger samples are better." The law of large numbers is the theorem that guarantees it.

After completing this notebook you will be able to:

- State the law of large numbers and its **i.i.d.** requirement.
- Reproduce the video's four-sided-die illustration (the 16-pair population, the running average).
- List the three conditions — random sampling, sufficient size, independence — and show each failing.
- Recognize LLN as the license behind every "estimate by simulation" used in this course.

## 2. Theory

### The experiment: a four-sided die, averaged in pairs

Take a fair four-sided die with outcomes $1, 2, 3, 4$ and mean $2.5$. Throw it **twice** and record the *average* of the pair. There are $16$ equally likely pairs, and the table of pair-averages runs from $1.0$ (for $1,1$) to $4.0$ (for $4,4$). The mean of all 16 outcomes is the population mean of this experiment: **2.5**.

### Drawing samples, one at a time

Now draw from this population of pair-averages repeatedly and track the **running average**:

- First draw: pair $(4, 3)$ → average $3.5$. Our first estimate $\bar{x}_1 = 3.5$ — quite far from 2.5.
- Two draws: $(3,4)$ and $(1,3)$ → running mean of $3.5$ and $2.0$ pulls the estimate toward the center.
- Three draws, four draws… the estimate keeps updating.
- By nine draws: $2.56$ — already pretty close to $2.5$.

The more samples we take, the closer we tend to get to the population mean.

### The law, formally

Let $X_1, X_2, \dots, X_n$ be samples from the population — **independent and identically distributed (i.i.d.)**, each with the same distribution as $X$. Then, as $n \to \infty$:

$$
\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i \;\longrightarrow\; E[X] = \mu
$$

The sample average converges to the population mean.

### The conditions

1. **Random sampling** — the samples must be drawn randomly from the population (a biased rule converges to the wrong number, confidently).
2. **Sufficiently large $n$** — the guarantee is about the limit; the larger the sample, the more accurate the sample mean is *likely* to be.
3. **Independence** — the individual observations must be independent of each other (dependent draws can keep the average wandering).

(There is also a hidden fourth condition, met by every distribution in this course: the population mean must *exist* — the Cauchy distribution from the moments notebook is the classic outlaw.)

## 3. Worked Examples

### Example 1: the 16-pair table

| avg | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| **1** | 1.0 | 1.5 | 2.0 | 2.5 |
| **2** | 1.5 | 2.0 | 2.5 | 3.0 |
| **3** | 2.0 | 2.5 | 3.0 | 3.5 |
| **4** | 2.5 | 3.0 | 3.5 | 4.0 |

Mean of all 16 entries: $\frac{40}{16} = 2.5$ — as it must be: the average of averages of a fair die is the die's mean.

### Example 2: the video's sequence

$\bar{x}_1 = 3.5$ (one pair), then with $(3,4)$ and $(1,3)$: $\bar{x}_2 = \frac{3.5 + 2.0}{2} = 2.75$; further draws wobble the estimate but with ever-smaller steps — each new observation moves an $n$-point average by at most $\frac{|x - \bar{x}|}{n+1}$. After 9 draws: $2.56$.

### Example 3: why this licenses simulation

Every "simulate 100,000 times and take the mean" in this course was a silent appeal to the LLN: the empirical average of i.i.d. simulated outcomes converges to the expectation being estimated. The law is the bridge between probability (expectations) and statistics (averages).

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

faces = np.array([1, 2, 3, 4])
print(f"four-sided die, mean = {faces.mean()}")

### The 16-pair population and its mean

In [ ]:
pair_averages = pd.DataFrame([[(a + b) / 2 for b in faces] for a in faces],
                             index=faces, columns=faces)
pair_averages.index.name = "throw 1"
pair_averages.columns.name = "throw 2"
print(pair_averages)
print(f"\nmean of all 16 pair-averages: {pair_averages.values.mean()}")

### The video's experiment: a running average of pair-draws

In [ ]:
rng = np.random.default_rng(10)

n_draws = 50
draws = rng.choice(faces, size=(n_draws, 2)).mean(axis=1)
running = np.cumsum(draws) / np.arange(1, n_draws + 1)

print("draw  pair-average  running mean")
for i in range(9):
    print(f"{i+1:>4} {draws[i]:>13.1f} {running[i]:>13.3f}")
print("...")
print(f"{n_draws:>4} {draws[-1]:>13.1f} {running[-1]:>13.3f}   (target: 2.5)")

### The law at work across very different distributions

Whatever the shape — flat, skewed, two-valued — the running average of i.i.d. draws heads to $\mu$.

In [ ]:
rng = np.random.default_rng(0)

n = 100_000
experiments = {
    "four-sided die (mu = 2.5)": (rng.choice([1, 2, 3, 4], n), 2.5),
    "fair coin 0/1 (mu = 0.5)": (rng.integers(0, 2, n).astype(float), 0.5),
    "exponential  (mu = 5.0)": (rng.exponential(5, n), 5.0),
    "pm100 game   (mu = 0.0)": (rng.choice([-100.0, 100.0], n), 0.0),
}

for name, (draws, mu) in experiments.items():
    running = np.cumsum(draws) / np.arange(1, n + 1)
    print(f"{name}: after 100 draws {running[99]:>8.3f}, "
          f"after 100,000 draws {running[-1]:>8.4f}   (mu = {mu})")

### When the conditions fail, the law fails

Three broken setups: a biased sampling rule, strongly dependent draws, and a distribution with no mean at all.

In [ ]:
rng = np.random.default_rng(1)

n = 100_000
population = rng.normal(160, 8, size=100_000)
mu = population.mean()

# 1. NOT random: only sample people taller than the median
tall_only = rng.choice(population[population > np.median(population)], size=n)
print(f"biased rule:      running mean -> {tall_only.mean():.2f}   (mu = {mu:.2f})")
print("converges -- but to the WRONG number: randomness is condition #1.\n")

# 2. NOT independent: each draw is 0.999 * previous + small noise
dependent = np.empty(n)
dependent[0] = 0.0
noise = rng.normal(0, 1, n)
for i in range(1, n):
    dependent[i] = 0.999 * dependent[i - 1] + noise[i]
run_dep = np.cumsum(dependent) / np.arange(1, n + 1)
print(f"dependent draws:  running mean after 10^5 draws = {run_dep[-1]:+.3f} (mu = 0)")
print("still far off -- correlated observations carry far less information each.\n")

# 3. No mean exists: Cauchy draws
cauchy = rng.standard_cauchy(n)
run_cauchy = np.cumsum(cauchy) / np.arange(1, n + 1)
print(f"Cauchy draws:     running mean keeps jumping (last value {run_cauchy[-1]:+.2f})")
print("no mu to converge to -- the hidden condition (E[X] must exist) matters.")

## 5. Visualizations

### The classic LLN picture: many running averages, one destination

Ten independent runs of the pair-average experiment: early chaos, late consensus at 2.5.

In [ ]:
rng = np.random.default_rng(4)

n = 2000
plt.figure(figsize=(10, 5))
for _ in range(10):
    draws = rng.choice(faces, size=(n, 2)).mean(axis=1)
    running = np.cumsum(draws) / np.arange(1, n + 1)
    plt.plot(running, linewidth=1, alpha=0.7)
plt.axhline(2.5, color="black", linestyle="--", linewidth=2, label="$\\mu$ = 2.5")
plt.xscale("log")
plt.title("Ten running averages of the pair-average experiment (log-scale n)")
plt.xlabel("number of draws n")
plt.ylabel("running sample mean")
plt.ylim(1.4, 3.6)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Same law, four distributions

The ±\$100 game takes visibly longer than the coin — convergence speed depends on the spread — but all four arrive.

In [ ]:
rng = np.random.default_rng(6)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
setups = [("four-sided die", lambda k: rng.choice([1, 2, 3, 4], k), 2.5),
          ("fair coin (0/1)", lambda k: rng.integers(0, 2, k).astype(float), 0.5),
          ("exponential(5)", lambda k: rng.exponential(5, k), 5.0),
          ("win/lose $100", lambda k: rng.choice([-100.0, 100.0], k), 0.0)]

for ax, (name, gen, mu) in zip(axes.flat, setups):
    for _ in range(5):
        d = gen(5000)
        ax.plot(np.cumsum(d) / np.arange(1, 5001), linewidth=0.9, alpha=0.7)
    ax.axhline(mu, color="black", linestyle="--", linewidth=1.5)
    ax.set_xscale("log")
    ax.set_title(f"{name}: converging to {mu}")
    ax.grid(True, alpha=0.3)
axes[1, 0].set_xlabel("n (log scale)")
axes[1, 1].set_xlabel("n (log scale)")
plt.suptitle("The law of large numbers does not care about the shape -- only about i.i.d.")
plt.tight_layout()
plt.show()

### The lawbreakers

Running averages under the three violated conditions — biased sampling converges to the wrong target, dependence crawls, Cauchy never settles.

In [ ]:
rng = np.random.default_rng(9)

n = 50_000
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# biased rule
tall = population[population > np.median(population)]
d = rng.choice(tall, size=n)
axes[0].plot(np.cumsum(d) / np.arange(1, n + 1), color="tab:orange")
axes[0].axhline(population.mean(), color="black", linestyle="--", label="true $\\mu$")
axes[0].set_title("biased sampling:\nconverges to the wrong number")
axes[0].legend()

# dependent draws
dep = np.empty(n)
dep[0] = 0
noise = rng.normal(0, 1, n)
for i in range(1, n):
    dep[i] = 0.999 * dep[i - 1] + noise[i]
axes[1].plot(np.cumsum(dep) / np.arange(1, n + 1), color="tab:purple")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("dependent draws:\nconvergence crawls")

# Cauchy
cau = rng.standard_cauchy(n)
axes[2].plot(np.cumsum(cau) / np.arange(1, n + 1), color="tab:red")
axes[2].axhline(0, color="black", linestyle="--")
axes[2].set_title("Cauchy (no mean):\nnever settles")

for ax in axes:
    ax.set_xlabel("n")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("running mean")
plt.suptitle("Break a condition, lose the law")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Toss a fair coin (heads = 1) 10, 100, 10,000 times by simulation and report the running proportion of heads at each checkpoint. What value is the law promising, and does your run keep the promise?

**Exercise 2.** In the pair-average experiment, what are the *possible values* of a single draw, and which are most likely? Why does the running average nevertheless converge to a value (2.5) that single draws hit only $\tfrac{4}{16}$ of the time?

### Intermediate

**Exercise 3.** Quantify "the larger, the more accurate": for the four-sided die, simulate the probability that $|\bar{X}_n - 2.5| > 0.1$ for $n = 10, 100, 1000, 10{,}000$. How does it decay?

**Exercise 4.** The casino's law: a roulette bet on red wins \$1 with probability $\frac{18}{38}$, loses \$1 otherwise ($\mu \approx -0.0526$). Simulate a gambler's average result per game over 10,000 games, and the casino's average across 1,000 gamblers. Who does the LLN work for, and why?

### Advanced

**Exercise 5.** Convergence speed and spread: for distributions with $\sigma = 1, 10, 100$ (use scaled coins), estimate how many draws are needed for the running mean to stay within ±0.1 of $\mu$. Confirm the $n \propto \sigma^2$ scaling — a preview of the central limit theorem's precision.

**Exercise 6.** LLN for other statistics: the video notes the phenomenon "happens with other metrics, not just expected value." Demonstrate it for the sample *variance* ($s^2 \to \sigma^2$) and the sample *median* (→ population median) on the exponential distribution — the whole point-estimation module in one convergence plot.

## 7. Solutions

In [ ]:
# Exercise 1: coin checkpoints
rng = np.random.default_rng(0)

flips = rng.integers(0, 2, 10_000)
running = np.cumsum(flips) / np.arange(1, 10_001)
for k in [10, 100, 10_000]:
    print(f"after {k:>6} flips: proportion of heads = {running[k - 1]:.4f}")
print("The law promises 0.5 in the limit -- each checkpoint is closer.")

In [ ]:
# Exercise 2: possible values of one pair-average
vals, counts = np.unique(pair_averages.values, return_counts=True)
for v, c in zip(vals, counts):
    print(f"pair-average {v}: {c}/16")
print("2.5 is the most likely single value (4/16) but still rare; the RUNNING")
print("average converges because errors above and below 2.5 are symmetric and")
print("cancel as they accumulate -- convergence is about the aggregate, not")
print("about individual draws hitting the target.")

In [ ]:
# Exercise 3: P(|mean - 2.5| > 0.1) decays
rng = np.random.default_rng(1)

for n in [10, 100, 1000, 10_000]:
    means = rng.choice([1, 2, 3, 4], size=(20_000, n)).mean(axis=1)
    print(f"n = {n:>6}: P(|X_bar - 2.5| > 0.1) = {np.mean(np.abs(means - 2.5) > 0.1):.4f}")
print("Roughly a factor ~10 smaller per factor 10 in n at first, then collapsing")
print("to zero: the miss probability vanishes as n grows.")

In [ ]:
# Exercise 4: the casino IS the law of large numbers
rng = np.random.default_rng(2)

p_win = 18 / 38
mu_bet = p_win * 1 + (1 - p_win) * (-1)

gambler = rng.choice([1.0, -1.0], p=[p_win, 1 - p_win], size=10_000)
print(f"Exercise 4: mu per bet = {mu_bet:.4f}")
print(f"one gambler, 10,000 games: average result = {gambler.mean():+.4f}")

casino = rng.choice([1.0, -1.0], p=[p_win, 1 - p_win], size=(1000, 10_000)).mean()
print(f"casino across 1,000 such gamblers: average = {casino:+.5f}")
print("The law works for whoever plays MANY i.i.d. rounds -- the gambler's own")
print("average also converges (to a loss!), and the casino, aggregating millions")
print("of bets, realizes mu almost exactly. The house edge is applied LLN.")

In [ ]:
# Exercise 5: draws needed scale like sigma^2
rng = np.random.default_rng(3)

print("Exercise 5:")
for sigma in [1, 10, 100]:
    # coin taking values +-sigma: mean 0, std sigma
    for n in [100, 1000, 10_000, 100_000, 1_000_000]:
        means = rng.choice([-sigma, sigma], size=(300, n)).mean(axis=1)
        if np.mean(np.abs(means) <= 0.1) >= 0.95:
            print(f"  sigma = {sigma:>3}: n ~ {n:>9,} for 95% within +-0.1")
            break
    else:
        print(f"  sigma = {sigma:>3}: > 1,000,000 draws needed")
print("Each 10x in sigma costs ~100x in n: precision scales as sigma^2/n.")

In [ ]:
# Exercise 6: LLN for variance and median
rng = np.random.default_rng(5)

n = 20_000
draws = rng.exponential(5, n)
ns = np.arange(2, n + 1)

running_var = [np.var(draws[:k], ddof=1) for k in range(2, n + 1, 200)]
running_med = [np.median(draws[:k]) for k in range(2, n + 1, 200)]
ks = list(range(2, n + 1, 200))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ks, running_var, color="tab:blue")
axes[0].axhline(25, color="black", linestyle="--", label="$\\sigma^2$ = 25")
axes[0].set_title("running sample variance $s^2$")
axes[1].plot(ks, running_med, color="tab:green")
axes[1].axhline(5 * np.log(2), color="black", linestyle="--",
                label="median = 5 ln 2")
axes[1].set_title("running sample median")
for ax in axes:
    ax.set_xlabel("n")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.suptitle("Exercise 6: sample statistics converge to their population targets")
plt.tight_layout()
plt.show()

print("Exercise 6: both estimators settle onto their targets -- the 'law' extends")
print("to variances, medians, proportions, quantiles: sample statistics converge")
print("to population values under i.i.d. sampling.")

## 8. Conclusion

- The **law of large numbers**: for i.i.d. samples $X_1, \dots, X_n$ distributed like $X$, the sample average $\bar{X}_n = \frac1n\sum X_i$ converges to the population mean $\mu = E[X]$ as $n \to \infty$.
- The video's illustration: the four-sided-die pair-average population (16 outcomes, mean 2.5) — running averages start noisy ($\bar{x}_1 = 3.5$) and settle onto 2.5 (2.56 after nine draws).
- The **conditions**: (1) random sampling, (2) sufficiently large $n$, (3) independent observations — plus the fine print that $E[X]$ must exist. Break any one and the law breaks: biased rules converge to the wrong number, dependence crawls, Cauchy never settles.
- Convergence speed depends on the spread ($\propto \sigma^2/n$ draws for a given precision) — spelled out precisely by the central limit theorem, coming soon.
- The LLN is the license behind every simulation-based estimate in this course, the reason casinos profit from tiny edges, and the promise that the point estimates of this module ($\bar{x}$, $\hat p$, $s^2$) actually reach their targets.